In [ ]:
# Monte Carlo Policy Gradient method for RL (REINFORCE algorithm)

In [ ]:
import gymnasium as gym
from gymnasium.spaces.utils import flatten, flatdim
import math
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from IPython.display import clear_output
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

In [ ]:
# Create a torch module that has the state space as input, and outputs a softmax of action probabilities
class PolicyModel(nn.Module):
    def __init__(self, state_dims: int, action_dims: int):
        super().__init__()
        
        self.linear = nn.Linear(state_dims, 128)
        self.linear2 = nn.Linear(128, action_dims)
    
    def forward(self, x: torch.tensor) -> torch.tensor:
        x = F.relu(self.linear(x))
        x = F.softmax(self.linear2(x), dim=-1)
        return x

In [ ]:
def make_frozen_lake(render: bool) -> gym.Env:
    is_slippery = True
    map_name = "4x4"
    render_mode = "rgb_array" if render else None
    env = gym.make('FrozenLake-v1', desc=None, map_name=map_name, is_slippery=is_slippery, render_mode=render_mode)
    return env

def make_blackjack(render: bool) -> gym.Env:
    render_mode = "rgb_array" if render else None
    env = gym.make('Blackjack-v1', natural=False, sab=False, render_mode=render_mode)
    return env

def make_taxi(render: bool) -> gym.Env:
    render_mode = "rgb_array" if render else None
    env = gym.make('Taxi-v3', render_mode=render_mode)
    return env

def make_cliffwalking(render: bool) -> gym.Env:
    render_mode = "rgb_array" if render else None
    env = gym.make('CliffWalking-v0', render_mode=render_mode)
    return env

def make_cartpole(render: bool) -> gym.Env:
    render_mode = "rgb_array" if render else None
    env = gym.make('CartPole-v1', render_mode=render_mode)
    return env

def make_lunarlander(render: bool) -> gym.Env:
    render_mode = "rgb_array" if render else None
    env = gym.make("LunarLander-v2", render_mode=render_mode)
    return env

def make_env(render: bool=False) -> gym.Env:
    return make_frozen_lake(render)

In [ ]:
def convert_observation(obs, env: gym.Env) -> torch.tensor:
    """Converts an integer observation to a tensor"""
    return torch.as_tensor(flatten(env.observation_space, obs), dtype=torch.float)

In [ ]:
# Execute running our a greedy policy based on Q

def evaluate(env: gym.Env, Q: PolicyModel, render = True) -> float:
    Q.eval()
    done = False
    G_t = 0

    S, info = env.reset()
    S = convert_observation(S, env)

    if render:
        clear_output(wait=True)
        plt.imshow(env.render())
        plt.show()

    while not done:
        actions = Q(S)
        actions = actions.detach().numpy()
        A = np.argmax(actions)
        S_p, reward, terminated, trunc, info = env.step(A)
        G_t += reward
        S_p = convert_observation(S_p, env)

        S = S_p

        if render:
            clear_output(wait=True)
            plt.imshow(env.render())
            plt.show()

        if terminated or trunc:
            done = True
    
    # return the total return
    return G_t

In [ ]:
# Sample an episode from the MDP
def sampleEpisode(policy: PolicyModel, env: gym.Env) -> list:
    obs, info = env.reset()
    episode = []
    accReward = 0
    while True:
        obs_t = convert_observation(obs, env)
        probs = policy(obs_t)
        dist = Categorical(probs)
        action = dist.sample()
        lp_action = dist.log_prob(action)
        
        nextObs, reward, terminated, trunc, info = env.step(action.item())
        accReward += reward
        episode.append((obs, action, lp_action, reward, nextObs, accReward))
        obs = nextObs
        if trunc or terminated:
            break
    return episode

In [ ]:
#sampleEpisode()

Policy gradient update expression

$\theta \leftarrow \theta + \alpha \nabla_{\theta} \log\pi_{\theta}(s_t, a_t) G_{t}$

See here for an example of the loss function
https://docs.pytorch.org/docs/stable/distributions.html

Note: The original REINFORCE algorithm is gradient ascent, but PyTorch optimisers use descent so we use -ve for the log likelihood

In [ ]:
# Hyperparameters
epochs = 10
episodesPerEpoch = 1000
gamma = 0.99

# Create the environment and policy
env = make_env()
print("Created Environment: ", env.spec.id)
print("Observation Dim: ", flatdim(env.observation_space))
print("Action Dim: ", env.action_space.n)

# Create policy model and optimiser
policy = PolicyModel(flatdim(env.observation_space), env.action_space.n)
optimizer = optim.Adam(params=policy.parameters(), lr=0.001)

obs, info = env.reset()

avg_losses = []
avg_epoch_returns = []

for epoch in range(epochs):
    print("Epoch", epoch)
    losses = []
    episode_returns = []
    for idx in tqdm(range(episodesPerEpoch)):
        #print("Episode: ", idx)
        episode = sampleEpisode(policy, env)
        episode_returns.append(episode[-1][-1])
        
        # Go through each step in the episode in reverse order
        # Accumulate the total loss for the episode, then do a single update
        G_t_1 = 0 # Intialise return from after the final step to 0
        episode_loss = 0
        returns = []
        for step in reversed(episode):
            o, a, lp_a, r, n_o, accR = step
            # G_t = r_t + gamma * G_t+1
            G_t = r + gamma * G_t_1
            
            #episode_loss += -lp_a * G_t
            returns.insert(0, G_t)
            
            # Update G_(t+1) for the next iteration
            G_t_1 = G_t
        
        for step_idx in range(len(episode)):
            o, a, lp_a, r, n_o, accR = episode[step_idx]
            episode_loss += -lp_a * returns[step_idx]

        #print(episode_loss.item())
        # Single backward pass and optimizer step per episode
        optimizer.zero_grad()
        episode_loss.backward()
        optimizer.step()

        losses.append(episode_loss.item())
        
    avg_losses.append(sum(losses) / len(losses))
    avg_epoch_returns.append(sum(episode_returns) / len(episode_returns))

In [ ]:
plt.plot(avg_losses)
plt.title("Average Losses")

In [ ]:
plt.plot(avg_epoch_returns)
plt.title("Average Returns")

In [ ]:
# Visualise a gym run
env2 = make_env(render=True)
G = evaluate(env2, policy, render=True)
print("Total Return: ", G)

In [ ]:
# Evaluation runs to determine final average reward
final_evaluation_run_count = 500

s = 0
for _ in tqdm(range(final_evaluation_run_count)):
    s += evaluate(env, policy, render=False)
av_return = s / final_evaluation_run_count
print("Average Return: ", av_return)